# ARS Generation Equivalence Debug (isolated)

A diagnostic copy for investigating why greedy outputs differ between batch-1 and larger batches. It does not collect RFT data, train adapters, calibrate deployment, or run the benchmark. Run it in a separate Kaggle session so existing V5/V2 sessions remain untouched.

Start with 20 questions; raise `N_QUESTIONS` to 100 after the short diagnostic. It compares batch sizes 2 and 4 against the same batch-1 outputs and saves mismatch examples.

In [ ]:
import os
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["ARS_ATTENTION_IMPLEMENTATION"] = "sdpa"

N_QUESTIONS = 20  # Increase to 100 for a stronger diagnostic
BATCH_SIZES = [2, 4]
OUTPUT_DIR = Path("report/generation_equivalence_debug")
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
BRANCH = "development"
print({"n_questions": N_QUESTIONS, "batch_sizes": BATCH_SIZES, "repo": str(REPO_DIR)}, flush=True)


In [ ]:
import subprocess
import sys

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Diagnostic checkout ready:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip(), flush=True)


## Load the frozen base model and a fixed prompt sample

This uses the same no-steering, greedy setup for every comparison. The same ordered questions are used for batch-2 and batch-4.

In [ ]:
import torch
from src.configs.config import RSCConfig
from src.data.dataset import gsm8k_to_cot
from src.evaluation.generation import equivalence_test
from scripts.run_ars_pipeline import build_model, prepare_dataset_split

if not torch.cuda.is_available():
    raise RuntimeError("Select a Kaggle GPU before running this diagnostic")
device = torch.device("cuda:0")
cfg = RSCConfig()
cfg.use_gate = False
model, tokenizer = build_model(cfg, device)
model.eval()
model.set_routing_override("none", cfg.fixed_layers)
_, raw_val = gsm8k_to_cot(tokenizer, cfg)
rows = prepare_dataset_split(raw_val.with_format(None), limit=N_QUESTIONS)
if len(rows) < N_QUESTIONS:
    print(f"Using {len(rows)} available questions", flush=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Loaded {len(rows)} fixed questions; attention={os.environ.get('ARS_ATTENTION_IMPLEMENTATION')}", flush=True)


## Compare batch sizes and save diagnostics

Each comparison regenerates the reference one-question-at-a-time outputs, then tests one larger batch size. The reports include accuracy delta, exact string identity, first differing token, stop reason, and sample mismatches.

In [ ]:
import json

reports = {}
for batch_size in BATCH_SIZES:
    print(f"\n=== Testing batch size {batch_size} ===", flush=True)
    report = equivalence_test(model, tokenizer, rows, device, cfg, n=len(rows), batch_size=batch_size)
    report_path = OUTPUT_DIR / f"equivalence_b{batch_size}.json"
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    reports[batch_size] = {key: report[key] for key in ("n", "delta_lenient_pp", "string_identity", "passed")}
    print({"batch_size": batch_size, **reports[batch_size], "report": str(report_path)}, flush=True)
print("\nDiagnostic summary:", json.dumps(reports, indent=2), flush=True)
print("Mismatch reports:", OUTPUT_DIR.resolve(), flush=True)


## How to interpret it

- If batch-2 is much closer than batch-4, larger padded batches are likely increasing divergence.
- Inspect `equivalence_b*.json` for the first differing token and stop reasons.
- This experiment identifies the pattern; it does not automatically claim a fix. Keep current production evaluation on batch-1 until a change is validated.